# Quickstart

This page builds a stratified compartment space end to end and queries it. Every
cell runs against the public `summer4` API and is executed when the
documentation is built.

A compartmental model needs three things: a set of compartments, flows between
them, and a solver. summer4 currently provides the **first** of those three. This
quickstart is therefore a tour of the compartment space alone — see
{doc}`../evaluation/feature-completeness` for what the other two will require.

## 1. Declare the axes

A `Property` is a named group of mutually exclusive traits. Nothing is a
compartment yet — a property is just a vocabulary.

In [ ]:
from summer4 import Property, PropertyMap

state = Property("state", ("S", "I", "R"))
age = Property("age", ("0-4", "5-9", "10+"))
severity = Property("severity", ("mild", "severe"))

print(state.traits)
print(state.trait("I"))

## 2. Build the compartment table

`PropertyMap.from_property` bootstraps a map whose compartments are one
property's traits. `stratify` returns a **new** map — maps are immutable, so the
original is still usable.

In [ ]:
base = PropertyMap.from_property(state)
assert base.size == 3

by_age = base.stratify(age)
assert by_age.size == 9
assert base.size == 3  # unchanged: stratify returns a new map

by_age

## 3. Stratify only part of the space

`where=` takes a selector. Only compartments where the selector is *true* are
split, so the table becomes **ragged**: the `severity` axis exists on the
infectious compartments and nowhere else.

In [ ]:
pmap = by_age.stratify(severity, where=state["I"])

# S x 3 ages + I x 3 ages x 2 severities + R x 3 ages
assert pmap.size == 3 + 6 + 3 == 12
assert pmap.n_properties == 3

for index, label in enumerate(pmap.labels()):
    print(f"{index:>3}  {label}")

## 4. Query with selectors

Selectors compose with `&`, `|` and `~`. `select` returns `int32` indices;
`mask` returns a boolean array of the same length as the table.

In [ ]:
young_infectious = pmap.select(state["I"] & age["0-4"])
assert young_infectious.tolist() == [3, 4]  # mild and severe

school_or_younger = pmap.select(age[("0-4", "5-9")])
assert school_or_younger.size == 8

single = pmap.select_one(state["S"] & age["10+"])
print("young infectious:", young_infectious)
print("one susceptible compartment:", single)

## 5. Reach the ragged holes explicitly

Because `severity` is absent on `S` and `R`, both `severity["mild"]` and
`~severity["mild"]` skip those rows. This is deliberate: absence is *unknown*,
not *false*. `.absent()` and `.present()` are the two-valued escape hatches.

In [ ]:
mild = set(pmap.select(severity["mild"]).tolist())
not_mild = set(pmap.select(~severity["mild"]).tolist())
no_severity = set(pmap.select(severity.absent()).tolist())

assert mild.isdisjoint(not_mild)
assert mild.isdisjoint(no_severity)
assert mild | not_mild | no_severity == set(range(pmap.size))

print("mild        ", sorted(mild))
print("not mild    ", sorted(not_mild))
print("no severity ", sorted(no_severity))

## 6. Group compartments for aggregation

`partition` splits the table by one property. `group_by` returns every existing
combination of several properties, skipping compartments that are missing any of
them.


In [ ]:
for trait, indices in pmap.partition(age).items():
    print(f"{trait.name:>5}  {indices.tolist()}")

print()
for traits, indices in pmap.group_by(state, severity).items():
    names = "/".join(trait.name for trait in traits)
    print(f"{names:<12} {indices.tolist()}")

`group_by(state, severity)` returns only the `I` rows, because `S` and `R` have
no `severity` code. That omission is the API telling you something true about the
model: there is no such thing as a mild susceptible.

## What comes next

With a `PropertyMap` in hand, {doc}`../user/08-flows` attaches named flows,
compiles a `CompiledModel`, and steps a JAX vector field. `euler` returns the
final state only; a trajectory and results object are still to come.
